# Análise da Evasão Escolar no Brasil

Notebook de análise exploratória desenvolvido para a Avaliação G2.

## 1. Introdução ao problema

A evasão escolar é um indicador educacional relevante, pois representa a saída de estudantes do sistema de ensino antes da conclusão de determinada etapa. O objetivo deste notebook é analisar padrões de evasão no Brasil a partir de variáveis regionais, sociais e educacionais.

## 2. Bibliotecas utilizadas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine

sns.set_theme(style="whitegrid")

## 3. Leitura dos dados

In [ ]:
df = pd.read_csv("../dados/simulacao_evasao_escolar_brasil.csv")
df.head()

## 4. Explicação da base de dados

A base contém informações simuladas sobre evasão escolar no Brasil, incluindo ano, semestre, região, UF, município, rede de ensino, série, matrículas, evasões, taxa de evasão, renda familiar, desempenho escolar, acesso à internet e nível de risco.

In [ ]:
df.info()
df.describe()

## 5. Limpeza e preparação

In [ ]:
df["data"] = pd.to_datetime(df["data"], errors="coerce")
df = df.drop_duplicates()
df = df.dropna(subset=["ano", "semestre", "regiao", "uf", "matriculados", "evasoes", "taxa_evasao"])
df["periodo"] = df["ano"].astype(str) + "/" + df["semestre"].astype(str)
df.isna().sum()

## 6. Engenharia de atributos

In [ ]:
df["taxa_evasao_calculada"] = (df["evasoes"] / df["matriculados"] * 100).round(2)
df["alerta"] = np.where(df["taxa_evasao"] >= 12, "Crítico", np.where(df["taxa_evasao"] >= 8, "Atenção", "Controlado"))
df[["matriculados", "evasoes", "taxa_evasao", "taxa_evasao_calculada", "alerta"]].head()

## 7. KPIs

In [ ]:
kpis = {
    "Total de matrículas": int(df["matriculados"].sum()),
    "Total de evasões": int(df["evasoes"].sum()),
    "Taxa média de evasão": round(df["taxa_evasao"].mean(), 2),
    "Desempenho médio": round(df["indice_desempenho"].mean(), 2),
    "Acesso médio à internet": round(df["acesso_internet"].mean(), 2),
    "Renda média familiar": round(df["renda_media_familiar"].mean(), 2)
}
kpis

## 8. Análise exploratória

In [ ]:
plt.figure(figsize=(10,5))
temporal = df.groupby("periodo", as_index=False)["taxa_evasao"].mean()
sns.lineplot(data=temporal, x="periodo", y="taxa_evasao", marker="o")
plt.title("Evolução temporal da taxa média de evasão")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
regiao = df.groupby("regiao", as_index=False)["taxa_evasao"].mean().sort_values("taxa_evasao", ascending=False)
sns.barplot(data=regiao, x="regiao", y="taxa_evasao")
plt.title("Taxa média de evasão por região")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.scatterplot(data=df, x="renda_media_familiar", y="taxa_evasao", hue="nivel_risco")
plt.title("Renda familiar x Taxa de evasão")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(data=df, x="rede_ensino", y="taxa_evasao")
plt.title("Distribuição da evasão por rede de ensino")
plt.tight_layout()
plt.show()

## 9. Correlação estatística

In [ ]:
corr = df[["taxa_evasao", "renda_media_familiar", "indice_desempenho", "acesso_internet", "matriculados", "evasoes"]].corr()
plt.figure(figsize=(10,6))
sns.heatmap(corr, annot=True, cmap="Blues")
plt.title("Matriz de correlação")
plt.tight_layout()
plt.show()

## 10. Persistência em banco SQLite

In [ ]:
engine = create_engine("sqlite:///../database/evasao.db")
df.to_sql("evasao_escolar", engine, if_exists="replace", index=False)
pd.read_sql("SELECT * FROM evasao_escolar LIMIT 5", engine)

## 11. Interpretação dos resultados

A análise permite comparar a evasão escolar entre regiões, redes de ensino e níveis de risco. Também permite observar relações entre indicadores sociais e educacionais, como renda familiar, acesso à internet e desempenho escolar. Essas variáveis ajudam a compreender possíveis fatores associados ao aumento da evasão.

## 12. Conclusão

O projeto demonstrou a aplicação de Python para análise exploratória, tratamento de dados, criação de KPIs, visualização e persistência em banco. O dashboard em Streamlit complementa a análise ao permitir filtros interativos e visualizações executivas para apoiar decisões educacionais.